# **1. Perkenalan Dataset**


Dataset yang digunakan adalah **Breast Cancer Wisconsin (Diagnostic)**.

Sumber dataset:
- Sumber primer: UCI Machine Learning Repository
  https://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+(Diagnostic)
- Sumber akses pada eksperimen ini: loader resmi `scikit-learn` (`load_breast_cancer`)
  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html

Karakteristik dataset:
- Jumlah sampel: 569
- Jumlah fitur: 30 fitur numerik
- Tipe tugas: klasifikasi biner
- Target: `0 = malignant`, `1 = benign`

Alasan pemilihan dataset:
1. Sumber data jelas dan terdokumentasi.
2. Struktur data tabular numerik rapi sehingga cocok untuk EDA dan preprocessing.
3. Cocok untuk baseline model klasifikasi klasik seperti RandomForest dan LogisticRegression.

# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [5]:
from pathlib import Path

import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [6]:
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "preprocessing" else current_dir
raw_file = project_root / "breast_cancer_raw.csv"

if raw_file.exists():
    df = pd.read_csv(raw_file)
else:
    dataset = load_breast_cancer(as_frame=True)
    df = dataset.frame.copy()
    raw_file.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(raw_file, index=False)

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [7]:
print("Shape:", df.shape)
print("\nInfo:")
df.info()
print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False).head())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget distribution:\n", df["target"].value_counts(normalize=True).sort_index())
print("\nDescriptive statistics:\n", df.describe().T.head())
print("\nCorrelation with target (top 10 absolute values):")
print(df.corr(numeric_only=True)["target"].abs().sort_values(ascending=False).head(10))#Type your code here

Shape: (569, 31)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness er

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [8]:
processed_df = df.drop_duplicates().copy()
feature_columns = [column for column in processed_df.columns if column != "target"]

# Handle missing values on numeric features with median to keep distribution robust.
processed_df[feature_columns] = processed_df[feature_columns].fillna(
    processed_df[feature_columns].median(numeric_only=True)
)

scaler = StandardScaler()
processed_df[feature_columns] = scaler.fit_transform(processed_df[feature_columns])

processed_output_file = project_root / "preprocessing" / "breast_cancer_preprocessed.csv"
processed_output_file.parent.mkdir(parents=True, exist_ok=True)
processed_df.to_csv(processed_output_file, index=False)

reloaded_processed = pd.read_csv(processed_output_file)
print("Processed file:", processed_output_file)
print("Raw shape:", df.shape)
print("Processed shape:", reloaded_processed.shape)
print("Processed missing values:", reloaded_processed.isna().sum().sum())
print("Processed duplicate rows:", reloaded_processed.duplicated().sum())

assert reloaded_processed.shape[0] == processed_df.shape[0]
assert reloaded_processed.columns.tolist() == processed_df.columns.tolist()
assert "target" in reloaded_processed.columns

print("Notebook preprocessing verification passed.")

Processed file: C:\Users\DianErdiana\dianerdiana-learning\machine-learning-dicoding\Eksperimen_SML_Dian-Erdiana\preprocessing\breast_cancer_preprocessed.csv
Raw shape: (569, 31)
Processed shape: (569, 31)
Processed missing values: 0
Processed duplicate rows: 0
Notebook preprocessing verification passed.
